In [40]:
import pandas as pd

In [41]:
df = pd.read_csv('adtech_click_log.csv')

In [42]:
df

,timestamp,ad_id,campaign_id,publisher_id,device_type,country,bid_price,clicked
0,2025-04-06 16:34:48,ad_39,camp_1,pub_12,mobile,US,2.46,0
1,2025-04-09 00:40:14,ad_29,camp_9,pub_8,tablet,US,1.91,0
2,2025-04-07 18:49:03,ad_15,camp_3,pub_4,tablet,UK,1.92,0
3,2025-04-10 05:56:34,ad_43,camp_9,pub_7,desktop,IN,2.88,0
4,2025-04-10 02:49:06,ad_8,camp_10,pub_3,mobile,IN,1.88,0
...,...,...,...,...,...,...,...,...
49995,2025-04-10 11:43:19,ad_26,camp_3,pub_5,mobile,US,4.88,0
49996,2025-04-05 16:09:31,ad_22,camp_10,pub_12,tablet,SG,2.05,0
49997,2025-04-11 11:14:31,ad_15,camp_7,pub_12,mobile,SG,1.66,0
49998,2025-04-11 16:49:04,ad_9,camp_6,pub_13,mobile,AU,2.02,0


CTR by country

In [43]:
country_stats = df.groupby('country').agg(
    impressions=('clicked','count'),
    clicks=('clicked','sum'),
    spend=('bid_price','sum'),
    bid_avg=('bid_price','mean')
)
country_stats['CTR'] = country_stats['clicks']/country_stats['impressions']
country_stats.sort_values('CTR', ascending=False)

,impressions,clicks,spend,bid_avg,CTR
country,,,,,
IN,20037,1356,55326.94,2.761239,0.067675
UK,7633,513,20994.62,2.750507,0.067208
SG,7436,484,20468.71,2.752651,0.065089
US,9915,588,27427.34,2.766247,0.059304
AU,4979,283,13661.10,2.743744,0.056839


CTR by device type

In [44]:
device_stats = df.groupby('device_type').agg(
    impressions=('clicked','count'),
    clicks=('clicked','sum'),
    spend=('bid_price','sum'),
    bid_avg=('bid_price','mean')
)
device_stats['CTR'] = device_stats['clicks']/device_stats['impressions']
device_stats.sort_values('CTR', ascending=False)

,impressions,clicks,spend,bid_avg,CTR
device_type,,,,,
mobile,29985,2144,82617.10,2.755281,0.071502
desktop,14960,812,41295.02,2.760362,0.054278
tablet,5055,268,13966.59,2.762926,0.053017


CTR with country and device type

In [45]:
combo = df.groupby(['country','device_type']).agg(
    impressions=('clicked','count'),
    clicks=('clicked','sum'),
    spend=('bid_price','sum')
)
combo['CTR'] = combo['clicks']/combo['impressions']
combo_sorted = combo.sort_values('CTR', ascending=False)
combo_sorted

,,impressions,clicks,spend,CTR
country,device_type,,,,
UK,mobile,4525,341,12437.81,0.075359
IN,mobile,12017,903,33206.65,0.075144
SG,tablet,739,52,2043.74,0.070365
US,mobile,5909,398,16295.90,0.067355
SG,mobile,4550,305,12473.54,0.067033
AU,mobile,2984,197,8203.20,0.066019
SG,desktop,2147,127,5951.43,0.059152
IN,desktop,5958,350,16448.08,0.058745
AU,tablet,496,29,1358.98,0.058468


In [46]:
overall_ctr = df['clicked'].mean()
overall_ctr

0.06448

Countries like IN, UK, and SG have good CTR (+6.5%) and mobile has higher CTR than desktop and tablet. 


Combinations where spend is high but CTR is below average (6.448%)

In [47]:
combo['spend_pc'] = combo['spend']/combo['impressions'] 
low_ctr_high_spend = combo[(combo['CTR'] < overall_ctr)].copy()
low_ctr_high_spend = low_ctr_high_spend.sort_values('spend', ascending=False)
low_ctr_high_spend.head(10)

,,impressions,clicks,spend,CTR,spend_pc
country,device_type,,,,,
IN,desktop,5958,350,16448.08,0.058745,2.760671
US,desktop,3027,144,8417.09,0.047572,2.780671
UK,desktop,2329,134,6379.50,0.057535,2.739158
SG,desktop,2147,127,5951.43,0.059152,2.771975
IN,tablet,2062,103,5672.21,0.049952,2.750829
AU,desktop,1499,57,4098.92,0.038025,2.734436
US,tablet,979,46,2714.35,0.046987,2.772574
UK,tablet,779,38,2177.31,0.048780,2.795006
AU,tablet,496,29,1358.98,0.058468,2.739879


All top low CTR high spend combos are either on desktop opr tablet

CTR for different ads

In [55]:
ad_stats = df.groupby('ad_id').agg(
    impressions=('clicked','count'),
    clicks=('clicked','sum'),
    spend=('bid_price','sum')
)
ad_stats['CTR'] = ad_stats['clicks']/ad_stats['impressions']
ad_stats.sort_values('CTR', ascending=False).head()
#High performing ads

,impressions,clicks,spend,CTR
ad_id,,,,
ad_8,993,85,2713.91,0.085599
ad_24,1000,78,2745.26,0.078000
ad_10,1006,76,2745.45,0.075547
ad_26,1055,79,2882.39,0.074882
ad_33,967,72,2610.41,0.074457


In [56]:
#Low performing ads
ad_stats.sort_values('CTR').head()

,impressions,clicks,spend,CTR
ad_id,,,,
ad_11,972,51,2691.47,0.052469
ad_49,972,51,2769.13,0.052469
ad_12,968,52,2640.63,0.053719
ad_20,956,53,2644.73,0.055439
ad_44,1026,57,2838.10,0.055556


In [50]:
df[['bid_price','clicked']].corr()

,bid_price,clicked
bid_price,1.000000,0.012111
clicked,0.012111,1.000000


very low correlation between bid_price and clicked so bid price is not influenceing clicks

In [51]:
device_stats

,impressions,clicks,spend,bid_avg,CTR
device_type,,,,,
desktop,14960,812,41295.02,2.760362,0.054278
mobile,29985,2144,82617.10,2.755281,0.071502
tablet,5055,268,13966.59,2.762926,0.053017


In [53]:
desktop_tablet_spend = device_stats.loc[['desktop','tablet'],'spend'].sum()
mobile_spend = device_stats.loc['mobile','spend']
# shift 10% of desktop+tablet spend
shift = 0.1 * desktop_tablet_spend
# spend per click for each device approx
spc = device_stats['spend']/device_stats['clicks']
spc

device_type
desktop    50.855936
mobile     38.534095
tablet     52.114142
dtype: float64

In [52]:
cpc_mobile = spc['mobile']
cpc_others = (device_stats.loc['desktop','spend'] + device_stats.loc['tablet','spend']) / (device_stats.loc['desktop','clicks'] + device_stats.loc['tablet','clicks'])
shift_clicks_gain = shift/cpc_mobile - shift/cpc_others
shift_clicks_gain

35.40964744586773

By reallocating from low CTR high CPC combinations (desktop and tablet) to mobile, 35 extra clicks gained.

Final Thoughts

Q1 Top drivers of CTR

1. Mobile users click on ads at a rate of 7.15 %, significantly higher than desktop (5.4 %) or tablet (5.3 %).

2. Best-Performing Markets
The strongest geographies for CTR are: India, United Kingdom, Singapore

3. Device + Market Sweet Spots
When we look at device and geography together, two segments stand out:
  a. UK × Mobile: 7.54 % CTR
  b. IN × Mobile: 7.51 % CTR

4. Best ad ad_8 CTR = 8.56 %; worst ad_11 = 5.25 %

Q2 Data-backed recommendations

A. Improve overall CTR

   1.Increase spending in mobile ads
    Shift at least 10 % of spend from desktop+tablet to mobile.
    In this data, that swap would cost the same but buy nearly 35 extra clicks.

   2.Clone what already works
   Learn from high performing like ad_8 / ad_24  and A/B-test those elements into the low performers      like (ad_11, ad_49).
   Use IN/UK/SG landing-page experience (fast, minimal, localised) as the template for US/AU.

B. Cut wasted ad spend
 1. Throttle low-CTR, high-spend device segments like IN-desktop/US-desktop/IN-tablet , you can pause; relocate to mobile or Give one last creative test; if it doesn’t improve, stop it.
 
 2. Clean out weak ads
Ads 11, 49, 12 are worse than the average ad but cost the same.
switch their budgets to our top-performing ads (top 25 %)(ad_8,ad_24,ad_10,ad_26,ad_33).
    